# Signal-Based Multi-Asset Hedge Payoff Analysis

このノートブックは、**オプションを使わず、既存のマルチアセット配分をシグナルで制御する**ことで、

- ヘッジなしのマルチアセットポートフォリオ
- **First responder** 付きポートフォリオ
- **Second responder** 付きポートフォリオ
- **First + Second combined** ポートフォリオ

を比較するための分析テンプレートです。

主な評価軸は以下です。

1. Absolute NAV と drawdown
2. ベース対比の relative hedge payoff
3. horizon別 DDC / drawdown coverage
4. horizon別 payoff ratio
5. quadratic / piecewise drawdown convexity
6. DDC map と convexity map
7. worst drawdown windows での event study

データは `yfinance` 経由で Yahoo Finance から取得します。`yfinance` は Yahoo Finance の公式SDKではなく、研究・教育目的の利用を前提としたオープンソースツールです。利用範囲は Yahoo Finance 側の規約も確認してください。


## 0. Methodological setup

ベース・ポートフォリオの forward return を

\[
R^0_{t,t+h}
\]

戦略 \(j\) の forward return を

\[
R^j_{t,t+h}
\]

とします。ベース対比の hedge payoff は、

\[
\pi^j_{t,h} = R^j_{t,t+h} - R^0_{t,t+h}
\]

です。ベースの損失を

\[
L^0_{t,h} = -R^0_{t,t+h}
\]

と置くと、drawdown convexity は、

\[
E[\pi^j_{t,h} \mid L^0_{t,h}]
\]

が \(L^0_{t,h}\) に対してどれだけ非線形・凸的に増えるかで評価します。

ここで使うシグナルは option payoff ではなく、**path-dependent avoided-loss payoff** を作るための exposure control です。したがって、測っている convexity は contractual gamma ではなく、realized / ex-post drawdown convexity です。


In [ ]:
# Optional: install missing packages inside the notebook environment.
# In a controlled research environment, you may prefer to install these packages outside the notebook.
import importlib.util
import subprocess
import sys

REQUIRED_PACKAGES = {
    "yfinance": "yfinance",
    "statsmodels": "statsmodels",
    "openpyxl": "openpyxl",
}

for pip_name, import_name in REQUIRED_PACKAGES.items():
    if importlib.util.find_spec(import_name) is None:
        print(f"Installing missing package: {pip_name}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])


In [ ]:
import warnings
warnings.filterwarnings("ignore")

from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import yfinance as yf

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)
pd.options.display.float_format = "{:.4f}".format


## 1. Configuration

以下のブロックだけ編集すれば、ユニバース・ウェイト・ヘッジ強度・transaction cost・評価horizonを変更できます。

デフォルトでは、Yahoo Financeで取得しやすい米国上場ETFを使います。`BASE_WEIGHTS` はヘッジなしマルチアセット配分、`DEFENSIVE_WEIGHTS` はリスクオフ時の退避先です。


In [ ]:
# -----------------------------
# Data universe
# -----------------------------
START_DATE = "2007-01-01"
END_DATE = None  # None = latest available from Yahoo Finance
INTERVAL = "1d"

TICKER_NAME_MAP = {
    "SPY": "US Equity",
    "EFA": "Developed ex-US Equity",
    "EEM": "Emerging Equity",
    "AGG": "US Aggregate Bond",
    "HYG": "High Yield Credit",
    "GLD": "Gold",
    "DBC": "Broad Commodities",
    "SHY": "1-3Y Treasury",
    "IEF": "7-10Y Treasury",
}

TICKERS = list(TICKER_NAME_MAP.keys())

# Base multi-asset allocation. Must sum to 1 after normalization.
BASE_WEIGHTS = {
    "SPY": 0.30,
    "EFA": 0.15,
    "EEM": 0.05,
    "AGG": 0.25,
    "HYG": 0.10,
    "GLD": 0.05,
    "DBC": 0.05,
    "SHY": 0.05,
    "IEF": 0.00,
}

# Defensive allocation used when risk exposure multiplier g_t goes toward 0.
DEFENSIVE_WEIGHTS = {
    "SPY": 0.00,
    "EFA": 0.00,
    "EEM": 0.00,
    "AGG": 0.00,
    "HYG": 0.00,
    "GLD": 0.10,
    "DBC": 0.00,
    "SHY": 0.70,
    "IEF": 0.20,
}

# Transaction cost applied to one-way turnover, in basis points.
TC_BPS = 2.0
PERIODS_PER_YEAR = 252

# -----------------------------
# First responder parameters
# -----------------------------
# CPPI/TIPP-like fast floor signal, measured on base NAV as a practical portfolio-level state variable.
CPPI_MULTIPLIER = 4.0
CPPI_FLOOR_PCT_OF_HWM = 0.75
CPPI_MIN_G = 0.15

# Fast drawdown brake.
FAST_DD_LOOKBACK = 20
FAST_DD_TRIGGER = 0.03
FAST_DD_FULL_BRAKE = 0.08
FAST_DD_MIN_G = 0.20

# Fast volatility targeting.
FAST_VOL_SPAN = 20
FAST_VOL_TARGET = 0.10
FAST_VOL_MIN_G = 0.20

# First responder signal combination.
# "min" = emergency brake style. "mean" = smoother, less conservative.
FIRST_COMBINATION_METHOD = "min"

# -----------------------------
# Second responder parameters
# -----------------------------
SECOND_MIN_G = 0.25
MOM_LOOKBACKS = [63, 126, 252]  # approx. 3M, 6M, 12M business-day windows
MA_LOOKBACK = 200
SECOND_UP_ALPHA = 0.08   # slow re-risking
SECOND_DOWN_ALPHA = 0.45 # faster de-risking, but slower than first responder

# -----------------------------
# Combined strategy parameters
# -----------------------------
LAMBDA_FIRST = 0.35
USE_FIRST_RESPONDER_VETO = True

# -----------------------------
# DDC / convexity evaluation
# -----------------------------
HORIZONS = {
    "1D": 1,
    "1W": 5,
    "1M": 21,
    "1Q": 63,
}

# DDC event definition.
# quantile mode is usually more robust than fixed thresholds across samples and asset mixes.
DDC_THRESHOLD_MODE = "quantile"  # "quantile" or "absolute"
DDC_LOSS_QUANTILE = 0.75
DDC_ABS_THRESHOLDS = {1: 0.01, 5: 0.025, 21: 0.05, 63: 0.08}

# Convexity knots for piecewise regression.
PIECEWISE_KNOT_Q1 = 0.75
PIECEWISE_KNOT_Q2 = 0.90

OUTPUT_DIR = Path("analysis_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)


## 2. Download prices from Yahoo Finance

`auto_adjust=True` を使い、分配金・分割調整後の `Close` を使います。実行環境や `yfinance` のバージョンにより列構造が変わることがあるため、MultiIndex / single-index の両方に対応しています。


In [ ]:
def extract_close_from_yfinance(raw: pd.DataFrame, tickers: Iterable[str]) -> pd.DataFrame:
    '''Extract adjusted close-like prices from yfinance output robustly.'''
    tickers = list(tickers)
    if raw is None or raw.empty:
        raise ValueError("yfinance returned an empty DataFrame. Check tickers, dates, or network access.")

    if isinstance(raw.columns, pd.MultiIndex):
        level0 = list(raw.columns.get_level_values(0).unique())
        level1 = list(raw.columns.get_level_values(1).unique())

        # Standard group_by='column': first level contains OHLC fields, second level contains tickers.
        if "Close" in level0:
            close = raw["Close"].copy()
        elif "Adj Close" in level0:
            close = raw["Adj Close"].copy()
        # Alternative group_by='ticker': first level contains tickers, second level contains OHLC fields.
        elif "Close" in level1:
            close = raw.xs("Close", level=1, axis=1).copy()
        elif "Adj Close" in level1:
            close = raw.xs("Adj Close", level=1, axis=1).copy()
        else:
            raise ValueError("Could not find Close or Adj Close in yfinance MultiIndex columns.")
    else:
        # Single-ticker case or flattened output.
        if "Close" in raw.columns:
            close = raw[["Close"]].copy()
        elif "Adj Close" in raw.columns:
            close = raw[["Adj Close"]].copy()
        else:
            raise ValueError("Could not find Close or Adj Close in yfinance columns.")
        if len(tickers) == 1:
            close.columns = tickers

    close = close.reindex(columns=tickers)
    close.index = pd.to_datetime(close.index).tz_localize(None)
    close = close.sort_index()
    close = close.replace([np.inf, -np.inf], np.nan)
    return close


def download_prices(tickers: Iterable[str], start: str, end: Optional[str] = None, interval: str = "1d") -> pd.DataFrame:
    '''Download adjusted close prices with yfinance.'''
    tickers = list(tickers)
    raw = yf.download(
        tickers=tickers,
        start=start,
        end=end,
        interval=interval,
        auto_adjust=True,
        actions=False,
        progress=False,
        threads=True,
        group_by="column",
    )
    prices = extract_close_from_yfinance(raw, tickers)

    # Keep only dates where all assets have a valid price after forward filling short gaps.
    prices = prices.ffill().dropna(how="any")

    if prices.empty:
        raise ValueError("No complete price history after cleaning. Use a later START_DATE or fewer tickers.")
    return prices


prices = download_prices(TICKERS, START_DATE, END_DATE, INTERVAL)
returns = prices.pct_change().replace([np.inf, -np.inf], np.nan).dropna(how="any")

coverage = pd.DataFrame({
    "ticker": prices.columns,
    "name": [TICKER_NAME_MAP.get(t, t) for t in prices.columns],
    "first_date": [prices[t].first_valid_index() for t in prices.columns],
    "last_date": [prices[t].last_valid_index() for t in prices.columns],
    "n_obs": [prices[t].dropna().shape[0] for t in prices.columns],
}).set_index("ticker")

print(f"Price sample: {prices.index.min().date()} to {prices.index.max().date()} | rows={len(prices):,}")
display(coverage)
prices.tail()


## 3. Core utility functions


In [ ]:
def normalize_weights(weights: Dict[str, float], columns: Iterable[str], name: str = "weights") -> pd.Series:
    columns = list(columns)
    w = pd.Series(weights, dtype=float).reindex(columns).fillna(0.0)
    total = w.sum()
    if not np.isfinite(total) or abs(total) < 1e-12:
        raise ValueError(f"{name} sums to zero or non-finite.")
    return w / total


def nav_from_returns(r: pd.Series, initial: float = 1.0) -> pd.Series:
    r = r.fillna(0.0)
    return initial * (1.0 + r).cumprod()


def drawdown(nav: pd.Series) -> pd.Series:
    return nav / nav.cummax() - 1.0


def forward_return(r: pd.Series, h: int) -> pd.Series:
    '''Forward h-period simple return, aligned to the start date.'''
    if h < 1:
        raise ValueError("h must be >= 1")
    return (1.0 + r).rolling(h).apply(np.prod, raw=True).shift(-(h - 1)) - 1.0


def asymmetric_smooth(signal: pd.Series, up_alpha: float = 0.10, down_alpha: float = 0.50) -> pd.Series:
    '''Smooth an exposure multiplier with faster de-risking than re-risking.'''
    x = signal.astype(float).copy().clip(0.0, 1.0).ffill().fillna(1.0)
    out = pd.Series(index=x.index, dtype=float)
    out.iloc[0] = x.iloc[0]
    for i in range(1, len(x)):
        prev = out.iloc[i - 1]
        target = x.iloc[i]
        alpha = down_alpha if target < prev else up_alpha
        out.iloc[i] = prev + alpha * (target - prev)
    return out.clip(0.0, 1.0)


def summary_stats(r: pd.Series, turnover: Optional[pd.Series] = None, exposure: Optional[pd.Series] = None,
                  periods_per_year: int = 252) -> Dict[str, float]:
    r = r.dropna()
    nav = nav_from_returns(r)
    dd = drawdown(nav)
    years = len(r) / periods_per_year
    total_return = nav.iloc[-1] - 1.0
    cagr = nav.iloc[-1] ** (1.0 / years) - 1.0 if years > 0 else np.nan
    vol = r.std() * np.sqrt(periods_per_year)
    sharpe = r.mean() / r.std() * np.sqrt(periods_per_year) if r.std() > 0 else np.nan
    downside = r.clip(upper=0).std() * np.sqrt(periods_per_year)
    sortino = (r.mean() * periods_per_year) / downside if downside > 0 else np.nan
    max_dd = dd.min()
    calmar = cagr / abs(max_dd) if max_dd < 0 else np.nan

    out = {
        "Total Return": total_return,
        "CAGR": cagr,
        "Ann Vol": vol,
        "Sharpe": sharpe,
        "Sortino": sortino,
        "Max DD": max_dd,
        "Calmar": calmar,
        "Worst 1D": r.min(),
        "Worst 1M": forward_return(r, 21).min(),
        "Worst 1Q": forward_return(r, 63).min(),
    }
    if turnover is not None:
        out["Ann Turnover"] = turnover.reindex(r.index).fillna(0.0).mean() * periods_per_year
    if exposure is not None:
        g = exposure.reindex(r.index).dropna()
        out["Avg Exposure g"] = g.mean()
        out["Min Exposure g"] = g.min()
        out["Pct Defensive g<0.5"] = (g < 0.5).mean()
    return out


def make_weights(g: pd.Series, w_base: pd.Series, w_def: pd.Series, returns_index: pd.Index) -> pd.DataFrame:
    '''Create daily weights using lagged signal g_{t-1}.'''
    g_lag = g.reindex(returns_index).shift(1).ffill().fillna(1.0).clip(0.0, 1.0)
    weights = pd.DataFrame(index=returns_index, columns=w_base.index, dtype=float)
    for col in w_base.index:
        weights[col] = g_lag * w_base[col] + (1.0 - g_lag) * w_def[col]
    return weights


def portfolio_returns_from_weights(asset_returns: pd.DataFrame, weights: pd.DataFrame, tc_bps: float = 0.0) -> Tuple[pd.Series, pd.Series]:
    '''Compute net returns and one-way turnover from daily weights.'''
    weights = weights.reindex(asset_returns.index).ffill()
    gross = (weights * asset_returns).sum(axis=1)
    turnover = weights.diff().abs().sum(axis=1).fillna(0.0)
    tc = turnover * tc_bps / 10000.0
    net = gross - tc
    return net, turnover


## 4. First responder / second responder signals

First responder は短期ショックに即応するシグナルとして、以下の最小値を使います。

- CPPI/TIPP-like floor signal
- fast drawdown brake
- fast volatility targeting

Second responder は中期bear marketを捉えるシグナルとして、3M/6M/12M trend と 200日移動平均レジームを使います。リスク復元は意図的に遅くしています。


In [ ]:
def cppi_tipp_signal(base_ret: pd.Series, multiplier: float = 4.0, floor_pct_of_hwm: float = 0.75,
                     min_g: float = 0.15, max_g: float = 1.0) -> pd.Series:
    base_nav = nav_from_returns(base_ret)
    floor = floor_pct_of_hwm * base_nav.cummax()
    cushion = ((base_nav - floor) / base_nav).clip(lower=0.0)
    g = (multiplier * cushion).clip(min_g, max_g)
    return g.rename("CPPI/TIPP")


def fast_drawdown_signal(base_ret: pd.Series, lookback: int = 20, trigger: float = 0.03,
                         full_brake: float = 0.08, min_g: float = 0.20) -> pd.Series:
    base_nav = nav_from_returns(base_ret)
    rolling_peak = base_nav.rolling(lookback, min_periods=1).max()
    dd = 1.0 - base_nav / rolling_peak
    cut = ((dd - trigger) / (full_brake - trigger)).clip(0.0, 1.0)
    g = 1.0 - (1.0 - min_g) * cut
    return g.clip(min_g, 1.0).rename("Fast DD brake")


def fast_vol_signal(base_ret: pd.Series, span: int = 20, target_vol: float = 0.10,
                    min_g: float = 0.20, periods_per_year: int = 252) -> pd.Series:
    realized_vol = base_ret.ewm(span=span, adjust=False).std() * np.sqrt(periods_per_year)
    g = (target_vol / realized_vol).clip(min_g, 1.0)
    return g.ffill().fillna(1.0).rename("Fast vol target")


def first_responder_signal(base_ret: pd.Series) -> pd.DataFrame:
    cppi = cppi_tipp_signal(
        base_ret,
        multiplier=CPPI_MULTIPLIER,
        floor_pct_of_hwm=CPPI_FLOOR_PCT_OF_HWM,
        min_g=CPPI_MIN_G,
    )
    dd = fast_drawdown_signal(
        base_ret,
        lookback=FAST_DD_LOOKBACK,
        trigger=FAST_DD_TRIGGER,
        full_brake=FAST_DD_FULL_BRAKE,
        min_g=FAST_DD_MIN_G,
    )
    vol = fast_vol_signal(
        base_ret,
        span=FAST_VOL_SPAN,
        target_vol=FAST_VOL_TARGET,
        min_g=FAST_VOL_MIN_G,
        periods_per_year=PERIODS_PER_YEAR,
    )

    components = pd.concat([cppi, dd, vol], axis=1)
    if FIRST_COMBINATION_METHOD == "min":
        g = components.min(axis=1)
    elif FIRST_COMBINATION_METHOD == "mean":
        g = components.mean(axis=1)
    else:
        raise ValueError("FIRST_COMBINATION_METHOD must be 'min' or 'mean'.")

    components["First responder g"] = g.clip(0.0, 1.0)
    return components


def second_responder_signal(base_ret: pd.Series) -> pd.DataFrame:
    base_nav = nav_from_returns(base_ret)
    component_frames = []

    for lb in MOM_LOOKBACKS:
        mom = base_nav / base_nav.shift(lb) - 1.0
        component_frames.append((mom > 0).astype(float).rename(f"Momentum {lb}d"))

    ma = base_nav.rolling(MA_LOOKBACK, min_periods=max(20, MA_LOOKBACK // 4)).mean()
    component_frames.append((base_nav > ma).astype(float).rename(f"MA {MA_LOOKBACK}d regime"))

    components = pd.concat(component_frames, axis=1).fillna(1.0)
    score = components.mean(axis=1).clip(0.0, 1.0)
    raw_g = SECOND_MIN_G + (1.0 - SECOND_MIN_G) * score
    g = asymmetric_smooth(raw_g, up_alpha=SECOND_UP_ALPHA, down_alpha=SECOND_DOWN_ALPHA)

    components["Second score"] = score
    components["Second responder raw g"] = raw_g
    components["Second responder g"] = g
    return components


def combined_signal(g_first: pd.Series, g_second: pd.Series, lambda_first: float = 0.35,
                    use_first_veto: bool = True) -> pd.Series:
    g_first = g_first.clip(0.0, 1.0)
    g_second = g_second.clip(0.0, 1.0)
    g = lambda_first * g_first + (1.0 - lambda_first) * g_second
    if use_first_veto:
        g = pd.Series(np.minimum(g, g_first), index=g.index)
    return g.clip(0.0, 1.0).rename("Combined g")


## 5. Build the four portfolios

ウェイトは、

\[
w^j_t = g^j_{t-1} w^0 + (1-g^j_{t-1})w^{def}
\]

として、シグナルを1日ラグさせて適用します。これにより look-ahead bias を避けます。


In [ ]:
w_base = normalize_weights(BASE_WEIGHTS, returns.columns, "BASE_WEIGHTS")
w_def = normalize_weights(DEFENSIVE_WEIGHTS, returns.columns, "DEFENSIVE_WEIGHTS")

base_ret = (returns * w_base).sum(axis=1).rename("No hedge")

first_components = first_responder_signal(base_ret)
second_components = second_responder_signal(base_ret)

g_first = first_components["First responder g"].reindex(returns.index).ffill().fillna(1.0)
g_second = second_components["Second responder g"].reindex(returns.index).ffill().fillna(1.0)
g_combined = combined_signal(g_first, g_second, LAMBDA_FIRST, USE_FIRST_RESPONDER_VETO).reindex(returns.index).ffill().fillna(1.0)

first_weights = make_weights(g_first, w_base, w_def, returns.index)
second_weights = make_weights(g_second, w_base, w_def, returns.index)
combined_weights = make_weights(g_combined, w_base, w_def, returns.index)

first_ret, first_turnover = portfolio_returns_from_weights(returns, first_weights, TC_BPS)
second_ret, second_turnover = portfolio_returns_from_weights(returns, second_weights, TC_BPS)
combined_ret, combined_turnover = portfolio_returns_from_weights(returns, combined_weights, TC_BPS)

returns_df = pd.concat([
    base_ret,
    first_ret.rename("First responder"),
    second_ret.rename("Second responder"),
    combined_ret.rename("Combined"),
], axis=1).dropna()

nav_df = returns_df.apply(nav_from_returns)
dd_df = nav_df.apply(drawdown)

rel_payoff_df = pd.DataFrame(index=nav_df.index)
for col in ["First responder", "Second responder", "Combined"]:
    rel_payoff_df[col] = nav_df[col] / nav_df["No hedge"] - 1.0

g_df = pd.concat([
    g_first.rename("First responder g"),
    g_second.rename("Second responder g"),
    g_combined.rename("Combined g"),
], axis=1).reindex(returns_df.index).ffill()

turnover_df = pd.concat([
    pd.Series(0.0, index=returns_df.index, name="No hedge"),
    first_turnover.rename("First responder"),
    second_turnover.rename("Second responder"),
    combined_turnover.rename("Combined"),
], axis=1).reindex(returns_df.index).fillna(0.0)

print("Base weights")
display(w_base.to_frame("weight"))
print("Defensive weights")
display(w_def.to_frame("weight"))
print("Strategy returns")
display(returns_df.tail())


## 6. Basic performance and payoff visualization


In [ ]:
stats = []
for col in returns_df.columns:
    exposure = None
    if col == "First responder":
        exposure = g_df["First responder g"]
    elif col == "Second responder":
        exposure = g_df["Second responder g"]
    elif col == "Combined":
        exposure = g_df["Combined g"]
    else:
        exposure = pd.Series(1.0, index=returns_df.index)
    stats.append(pd.Series(summary_stats(returns_df[col], turnover_df[col], exposure, PERIODS_PER_YEAR), name=col))

stats_df = pd.DataFrame(stats)
display(stats_df.style.format("{:.2%}", subset=[c for c in stats_df.columns if c not in ["Sharpe", "Sortino", "Calmar"]])
        .format("{:.2f}", subset=["Sharpe", "Sortino", "Calmar"]))

stats_df.to_csv(OUTPUT_DIR / "performance_summary.csv")


In [ ]:
plt.figure(figsize=(12, 6))
nav_df.plot(ax=plt.gca())
plt.title("Absolute NAV: No hedge vs signal-based responders")
plt.ylabel("NAV")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 6))
dd_df.plot(ax=plt.gca())
plt.title("Drawdowns")
plt.ylabel("Drawdown")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 6))
rel_payoff_df.plot(ax=plt.gca())
plt.axhline(0.0, linestyle="--", linewidth=1)
plt.title("Relative hedge payoff vs No hedge")
plt.ylabel("Relative NAV payoff: Strategy / No hedge - 1")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 5))
g_df.plot(ax=plt.gca())
plt.title("Exposure multipliers g_t")
plt.ylabel("Risk-on multiplier")
plt.ylim(-0.05, 1.05)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 7. Drawdown coverage / DDC metrics

DDC は以下で定義します。

\[
DDC^j_h(\theta) = P(\pi^j_{t,h} > 0 \mid L^0_{t,h} > \theta_h)
\]

where

\[
\pi^j_{t,h} = R^j_{t,t+h} - R^0_{t,t+h}, \quad L^0_{t,h} = -R^0_{t,t+h}
\]

Payoff ratio は、

\[
\frac{E[\pi^j_{t,h} \mid L^0_{t,h}>\theta_h]}{E[L^0_{t,h} \mid L^0_{t,h}>\theta_h]}
\]

です。DDCが頻度、payoff ratio が大きさを表します。


In [ ]:
def ddc_metrics(strategy_returns: pd.Series, base_returns: pd.Series,
                horizons: Dict[str, int], threshold_mode: str = "quantile",
                loss_quantile: float = 0.75,
                abs_thresholds: Optional[Dict[int, float]] = None) -> pd.DataFrame:
    rows = []
    abs_thresholds = abs_thresholds or {}

    for label, h in horizons.items():
        rb = forward_return(base_returns, h)
        rs = forward_return(strategy_returns, h)
        payoff = rs - rb
        loss = -rb
        df = pd.DataFrame({"base_return": rb, "strategy_return": rs, "payoff": payoff, "loss": loss}).dropna()
        df = df[np.isfinite(df).all(axis=1)]
        loss_positive = df.loc[df["loss"] > 0, "loss"]

        if threshold_mode == "quantile":
            threshold = loss_positive.quantile(loss_quantile) if len(loss_positive) else np.nan
        elif threshold_mode == "absolute":
            threshold = abs_thresholds.get(h, np.nan)
        else:
            raise ValueError("threshold_mode must be 'quantile' or 'absolute'.")

        event = df["loss"] > threshold
        sub = df.loc[event]

        if len(sub) == 0:
            rows.append({
                "Horizon": label, "h": h, "Threshold": threshold, "N Events": 0,
                "DDC": np.nan, "Avg Hedge Payoff": np.nan, "Payoff Ratio": np.nan,
                "Avg Base Return in Event": np.nan, "Avg Strategy Return in Event": np.nan,
            })
            continue

        avg_payoff = sub["payoff"].mean()
        avg_loss = sub["loss"].mean()
        rows.append({
            "Horizon": label,
            "h": h,
            "Threshold": threshold,
            "N Events": int(len(sub)),
            "DDC": (sub["payoff"] > 0).mean(),
            "Avg Hedge Payoff": avg_payoff,
            "Payoff Ratio": avg_payoff / avg_loss if avg_loss > 0 else np.nan,
            "Avg Base Return in Event": sub["base_return"].mean(),
            "Avg Strategy Return in Event": sub["strategy_return"].mean(),
        })
    return pd.DataFrame(rows)


strategies = ["First responder", "Second responder", "Combined"]

ddc_all = []
for strategy in strategies:
    table = ddc_metrics(
        returns_df[strategy],
        returns_df["No hedge"],
        HORIZONS,
        threshold_mode=DDC_THRESHOLD_MODE,
        loss_quantile=DDC_LOSS_QUANTILE,
        abs_thresholds=DDC_ABS_THRESHOLDS,
    )
    table.insert(0, "Strategy", strategy)
    ddc_all.append(table)

ddc_df = pd.concat(ddc_all, ignore_index=True)
display(ddc_df)
ddc_df.to_csv(OUTPUT_DIR / "ddc_metrics.csv", index=False)


In [ ]:
# DDC by horizon
pivot_ddc = ddc_df.pivot(index="Horizon", columns="Strategy", values="DDC").reindex(HORIZONS.keys())
plt.figure(figsize=(10, 5))
pivot_ddc.plot(kind="bar", ax=plt.gca())
plt.title("DDC by horizon")
plt.ylabel("DDC: P(relative payoff > 0 | base loss event)")
plt.ylim(0, 1.05)
plt.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

# Payoff ratio by horizon
pivot_pr = ddc_df.pivot(index="Horizon", columns="Strategy", values="Payoff Ratio").reindex(HORIZONS.keys())
plt.figure(figsize=(10, 5))
pivot_pr.plot(kind="bar", ax=plt.gca())
plt.title("Payoff ratio by horizon")
plt.ylabel("Avg relative payoff / Avg base loss")
plt.axhline(0.0, linestyle="--", linewidth=1)
plt.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

# DDC map: x = Quarterly DDC demeaned, y = Daily DDC
map_rows = []
for strategy in strategies:
    sub = ddc_df[ddc_df["Strategy"] == strategy]
    daily = sub.loc[sub["Horizon"] == "1D", "DDC"].iloc[0]
    quarterly = sub.loc[sub["Horizon"] == "1Q", "DDC"].iloc[0]
    map_rows.append({"Strategy": strategy, "Daily DDC": daily, "Quarterly DDC": quarterly})

ddc_map = pd.DataFrame(map_rows)
ddc_map["Quarterly DDC demeaned"] = ddc_map["Quarterly DDC"] - ddc_map["Quarterly DDC"].mean()
display(ddc_map)

plt.figure(figsize=(8, 6))
plt.scatter(ddc_map["Quarterly DDC demeaned"], ddc_map["Daily DDC"], s=120)
for _, row in ddc_map.iterrows():
    plt.annotate(row["Strategy"], (row["Quarterly DDC demeaned"], row["Daily DDC"]), xytext=(6, 6), textcoords="offset points")
plt.axvline(0.0, linestyle="--", linewidth=1)
plt.title("DDC map: first vs second responder profile")
plt.xlabel("Quarterly DDC demeaned")
plt.ylabel("Daily DDC")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 8. Drawdown convexity evaluation

### 8.1 Quadratic convexity

\[
\pi^j_{t,h} = \alpha^j_h + \beta^j_h L^0_{t,h} + \gamma^j_h (L^0_{t,h})^2 + \varepsilon_{t,h}
\]

\(\gamma>0\) なら、大きな下落ほど relative hedge payoff が加速していると解釈できます。

### 8.2 Piecewise convexity

\[
\pi^j_{t,h} = \alpha + \beta_1 L^0_{t,h} + \beta_2(L^0_{t,h}-k_1)^+ + \beta_3(L^0_{t,h}-k_2)^+ + \varepsilon_{t,h}
\]

深い損失bucketで slope が上がるか、すなわち

\[
\beta_2 + \beta_3 > 0
\]

を見る方が、signal-based de-risking のような飽和型payoffには実務的です。


In [ ]:
def _loss_payoff_frame(strategy_returns: pd.Series, base_returns: pd.Series, h: int, loss_only: bool = True) -> pd.DataFrame:
    rb = forward_return(base_returns, h)
    rs = forward_return(strategy_returns, h)
    df = pd.DataFrame({
        "base_return": rb,
        "strategy_return": rs,
        "payoff": rs - rb,
        "loss": -rb,
    }).dropna()
    df = df[np.isfinite(df).all(axis=1)]
    if loss_only:
        df = df[df["loss"] > 0]
    return df


def quadratic_convexity_regression(strategy_returns: pd.Series, base_returns: pd.Series, h: int, min_obs: int = 40) -> Dict[str, float]:
    df = _loss_payoff_frame(strategy_returns, base_returns, h, loss_only=True)
    if len(df) < min_obs:
        return {"h": h, "Alpha": np.nan, "Beta": np.nan, "Gamma": np.nan, "t(Gamma)": np.nan, "R2": np.nan, "N": len(df)}

    df["loss2"] = df["loss"] ** 2
    X = sm.add_constant(df[["loss", "loss2"]])
    model = sm.OLS(df["payoff"], X).fit(cov_type="HAC", cov_kwds={"maxlags": max(1, h)})
    return {
        "h": h,
        "Alpha": model.params.get("const", np.nan),
        "Beta": model.params.get("loss", np.nan),
        "Gamma": model.params.get("loss2", np.nan),
        "t(Gamma)": model.tvalues.get("loss2", np.nan),
        "R2": model.rsquared,
        "N": int(model.nobs),
    }


def piecewise_convexity_regression(strategy_returns: pd.Series, base_returns: pd.Series, h: int,
                                   q1: float = 0.75, q2: float = 0.90,
                                   min_obs: int = 40, return_model: bool = False):
    df = _loss_payoff_frame(strategy_returns, base_returns, h, loss_only=True)
    if len(df) < min_obs:
        result = {
            "h": h, "k1": np.nan, "k2": np.nan,
            "Base Slope": np.nan, "Mid Loss Slope": np.nan, "Tail Loss Slope": np.nan,
            "Incremental Convexity": np.nan, "b2": np.nan, "b3": np.nan,
            "t(b2)": np.nan, "t(b3)": np.nan, "R2": np.nan, "N": len(df),
        }
        return (result, None, df) if return_model else result

    k1 = df["loss"].quantile(q1)
    k2 = df["loss"].quantile(q2)
    df["hinge1"] = np.maximum(df["loss"] - k1, 0.0)
    df["hinge2"] = np.maximum(df["loss"] - k2, 0.0)

    X = sm.add_constant(df[["loss", "hinge1", "hinge2"]])
    model = sm.OLS(df["payoff"], X).fit(cov_type="HAC", cov_kwds={"maxlags": max(1, h)})

    b1 = model.params.get("loss", np.nan)
    b2 = model.params.get("hinge1", np.nan)
    b3 = model.params.get("hinge2", np.nan)

    result = {
        "h": h,
        "k1": k1,
        "k2": k2,
        "Base Slope": b1,
        "Mid Loss Slope": b1 + b2,
        "Tail Loss Slope": b1 + b2 + b3,
        "Incremental Convexity": b2 + b3,
        "b2": b2,
        "b3": b3,
        "t(b2)": model.tvalues.get("hinge1", np.nan),
        "t(b3)": model.tvalues.get("hinge2", np.nan),
        "R2": model.rsquared,
        "N": int(model.nobs),
    }
    return (result, model, df) if return_model else result


def tail_payoff_ratio_table(strategy_returns: pd.Series, base_returns: pd.Series, h: int,
                            quantiles=(0.50, 0.75, 0.90, 0.95)) -> pd.DataFrame:
    df = _loss_payoff_frame(strategy_returns, base_returns, h, loss_only=True)
    rows = []
    for q in quantiles:
        if len(df) == 0:
            rows.append({"h": h, "Loss Quantile": q, "Loss Threshold": np.nan, "N Events": 0,
                         "Avg Payoff": np.nan, "Avg Loss": np.nan, "Tail Payoff Ratio": np.nan,
                         "Coverage": np.nan})
            continue
        k = df["loss"].quantile(q)
        sub = df[df["loss"] > k]
        avg_payoff = sub["payoff"].mean()
        avg_loss = sub["loss"].mean()
        rows.append({
            "h": h,
            "Loss Quantile": q,
            "Loss Threshold": k,
            "N Events": len(sub),
            "Avg Payoff": avg_payoff,
            "Avg Loss": avg_loss,
            "Tail Payoff Ratio": avg_payoff / avg_loss if avg_loss > 0 else np.nan,
            "Coverage": (sub["payoff"] > 0).mean() if len(sub) else np.nan,
        })
    return pd.DataFrame(rows)


def conditional_payoff_buckets(strategy_returns: pd.Series, base_returns: pd.Series, h: int,
                               quantiles=(0.50, 0.75, 0.90, 0.95, 1.00)) -> pd.DataFrame:
    df = _loss_payoff_frame(strategy_returns, base_returns, h, loss_only=True)
    if len(df) == 0:
        return pd.DataFrame()

    qs = [0.0] + list(quantiles)
    cuts = df["loss"].quantile(qs).values
    rows = []
    for i in range(len(cuts) - 1):
        lo, hi = cuts[i], cuts[i + 1]
        if i == len(cuts) - 2:
            sub = df[(df["loss"] >= lo) & (df["loss"] <= hi)]
        else:
            sub = df[(df["loss"] >= lo) & (df["loss"] < hi)]
        avg_loss = sub["loss"].mean()
        avg_payoff = sub["payoff"].mean()
        rows.append({
            "h": h,
            "Bucket": f"{qs[i]:.0%}-{qs[i+1]:.0%}",
            "Loss Low": lo,
            "Loss High": hi,
            "N": len(sub),
            "Avg Loss": avg_loss,
            "Avg Payoff": avg_payoff,
            "Payoff Ratio": avg_payoff / avg_loss if avg_loss > 0 else np.nan,
            "Coverage": (sub["payoff"] > 0).mean() if len(sub) else np.nan,
        })
    return pd.DataFrame(rows)


In [ ]:
quad_rows = []
piece_rows = []
tail_rows = []
bucket_rows = []

for strategy in strategies:
    for label, h in HORIZONS.items():
        qres = quadratic_convexity_regression(returns_df[strategy], returns_df["No hedge"], h)
        qres.update({"Strategy": strategy, "Horizon": label})
        quad_rows.append(qres)

        pres = piecewise_convexity_regression(
            returns_df[strategy], returns_df["No hedge"], h,
            q1=PIECEWISE_KNOT_Q1, q2=PIECEWISE_KNOT_Q2,
        )
        pres.update({"Strategy": strategy, "Horizon": label})
        piece_rows.append(pres)

        tpr = tail_payoff_ratio_table(returns_df[strategy], returns_df["No hedge"], h)
        tpr.insert(0, "Strategy", strategy)
        tpr.insert(1, "Horizon", label)
        tail_rows.append(tpr)

        buckets = conditional_payoff_buckets(returns_df[strategy], returns_df["No hedge"], h)
        if not buckets.empty:
            buckets.insert(0, "Strategy", strategy)
            buckets.insert(1, "Horizon", label)
            bucket_rows.append(buckets)

quad_convexity_df = pd.DataFrame(quad_rows)
piecewise_convexity_df = pd.DataFrame(piece_rows)
tail_payoff_ratio_df = pd.concat(tail_rows, ignore_index=True)
bucket_payoff_df = pd.concat(bucket_rows, ignore_index=True) if bucket_rows else pd.DataFrame()

cols_quad = ["Strategy", "Horizon", "h", "Beta", "Gamma", "t(Gamma)", "R2", "N"]
cols_piece = ["Strategy", "Horizon", "h", "Base Slope", "Mid Loss Slope", "Tail Loss Slope", "Incremental Convexity", "b2", "b3", "t(b2)", "t(b3)", "R2", "N"]

display(quad_convexity_df[cols_quad])
display(piecewise_convexity_df[cols_piece])

quad_convexity_df.to_csv(OUTPUT_DIR / "quadratic_convexity.csv", index=False)
piecewise_convexity_df.to_csv(OUTPUT_DIR / "piecewise_convexity.csv", index=False)
tail_payoff_ratio_df.to_csv(OUTPUT_DIR / "tail_payoff_ratio.csv", index=False)
bucket_payoff_df.to_csv(OUTPUT_DIR / "conditional_payoff_buckets.csv", index=False)


In [ ]:
# Horizon-by-horizon convexity score: Tail slope - Base slope.
pivot_conv = piecewise_convexity_df.pivot(index="Horizon", columns="Strategy", values="Incremental Convexity").reindex(HORIZONS.keys())
plt.figure(figsize=(10, 5))
pivot_conv.plot(kind="bar", ax=plt.gca())
plt.axhline(0.0, linestyle="--", linewidth=1)
plt.title("Piecewise drawdown convexity by horizon")
plt.ylabel("Incremental convexity: tail slope - base slope")
plt.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

# Convexity map: x = 1Q convexity, y = 1D convexity.
conv_map_rows = []
for strategy in strategies:
    sub = piecewise_convexity_df[piecewise_convexity_df["Strategy"] == strategy]
    c1d = sub.loc[sub["Horizon"] == "1D", "Incremental Convexity"].iloc[0]
    c1q = sub.loc[sub["Horizon"] == "1Q", "Incremental Convexity"].iloc[0]
    conv_map_rows.append({"Strategy": strategy, "1D Convexity": c1d, "1Q Convexity": c1q})

conv_map = pd.DataFrame(conv_map_rows)
display(conv_map)

plt.figure(figsize=(8, 6))
plt.scatter(conv_map["1Q Convexity"], conv_map["1D Convexity"], s=120)
for _, row in conv_map.iterrows():
    plt.annotate(row["Strategy"], (row["1Q Convexity"], row["1D Convexity"]), xytext=(6, 6), textcoords="offset points")
plt.axhline(0.0, linestyle="--", linewidth=1)
plt.axvline(0.0, linestyle="--", linewidth=1)
plt.title("Convexity map: short-horizon vs quarterly drawdown convexity")
plt.xlabel("1Q incremental convexity")
plt.ylabel("1D incremental convexity")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 9. Payoff-vs-loss scatter and fitted piecewise curve

下のチャートは、横軸にベースの損失 \(L^0\)、縦軸に戦略の相対payoff \(\pi^j\) を置きます。

- 右上がり：下落時にpayoffが出る
- 傾きがtailで増加：drawdown convexity
- 上に曲がるが飽和：dynamic de-risking 型の特徴
- 右下がり：crisis時に逆効果


In [ ]:
def plot_payoff_scatter_with_piecewise_fit(strategy: str, horizon_label: str, max_points: int = 2500):
    h = HORIZONS[horizon_label]
    result, model, df = piecewise_convexity_regression(
        returns_df[strategy], returns_df["No hedge"], h,
        q1=PIECEWISE_KNOT_Q1, q2=PIECEWISE_KNOT_Q2,
        return_model=True,
    )
    if df.empty or model is None:
        print(f"Not enough observations for {strategy}, {horizon_label}")
        return

    plot_df = df.copy()
    if len(plot_df) > max_points:
        plot_df = plot_df.sample(max_points, random_state=42).sort_values("loss")

    x_grid = np.linspace(df["loss"].min(), df["loss"].max(), 200)
    k1, k2 = result["k1"], result["k2"]
    X_grid = pd.DataFrame({
        "const": 1.0,
        "loss": x_grid,
        "hinge1": np.maximum(x_grid - k1, 0.0),
        "hinge2": np.maximum(x_grid - k2, 0.0),
    })
    y_fit = model.predict(X_grid)

    plt.figure(figsize=(8, 6))
    plt.scatter(plot_df["loss"], plot_df["payoff"], alpha=0.30, s=15)
    plt.plot(x_grid, y_fit, linewidth=2, label="Piecewise fitted payoff")
    plt.axhline(0.0, linestyle="--", linewidth=1)
    plt.axvline(k1, linestyle=":", linewidth=1, label=f"k1={k1:.2%}")
    plt.axvline(k2, linestyle=":", linewidth=1, label=f"k2={k2:.2%}")
    plt.title(f"Payoff vs base loss: {strategy}, horizon={horizon_label}")
    plt.xlabel("Base loss: -R0")
    plt.ylabel("Relative hedge payoff: R_strategy - R0")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


# Run representative charts.
for h_label in ["1D", "1Q"]:
    for strategy in strategies:
        plot_payoff_scatter_with_piecewise_fit(strategy, h_label)


## 10. Tail payoff ratio and conditional payoff buckets

Convexity があれば、より深い損失thresholdに条件付けるほど `Tail Payoff Ratio` が上がる傾向が出ます。


In [ ]:
# Tail payoff ratio curves by horizon.
for h_label in HORIZONS.keys():
    sub = tail_payoff_ratio_df[tail_payoff_ratio_df["Horizon"] == h_label]
    pivot = sub.pivot(index="Loss Quantile", columns="Strategy", values="Tail Payoff Ratio")
    plt.figure(figsize=(9, 5))
    pivot.plot(marker="o", ax=plt.gca())
    plt.axhline(0.0, linestyle="--", linewidth=1)
    plt.title(f"Tail payoff ratio by threshold: horizon={h_label}")
    plt.ylabel("E[payoff | loss>k] / E[loss | loss>k]")
    plt.xlabel("Base loss quantile threshold")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

# Conditional payoff buckets for selected horizon.
SELECTED_BUCKET_HORIZON = "1Q"
sub_bucket = bucket_payoff_df[bucket_payoff_df["Horizon"] == SELECTED_BUCKET_HORIZON]
if not sub_bucket.empty:
    pivot_bucket = sub_bucket.pivot(index="Bucket", columns="Strategy", values="Payoff Ratio")
    plt.figure(figsize=(10, 5))
    pivot_bucket.plot(kind="bar", ax=plt.gca())
    plt.axhline(0.0, linestyle="--", linewidth=1)
    plt.title(f"Conditional payoff ratio by base-loss bucket: horizon={SELECTED_BUCKET_HORIZON}")
    plt.ylabel("Avg payoff / Avg base loss")
    plt.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()

display(tail_payoff_ratio_df.head(20))


## 11. Worst drawdown window event study

手動で危機期間を指定する代わりに、ベース・ポートフォリオの worst forward return windows を自動抽出します。これにより、サンプルに依存した実際の悪化局面で、どのresponderがpayoffを出していたかを確認できます。


In [ ]:
def worst_forward_windows(base_returns: pd.Series, h: int = 63, top_n: int = 8, min_gap: int = 63) -> pd.DataFrame:
    fwd = forward_return(base_returns, h).dropna().sort_values()
    selected = []
    selected_dates = []
    for dt, val in fwd.items():
        if all(abs((dt - sdt).days) > min_gap for sdt in selected_dates):
            selected_dates.append(dt)
            selected.append({
                "start": dt,
                "end": base_returns.index[min(base_returns.index.get_loc(dt) + h - 1, len(base_returns.index) - 1)],
                "h": h,
                "base_forward_return": val,
            })
        if len(selected) >= top_n:
            break
    return pd.DataFrame(selected)


def plot_event_window(start_date, pre_days: int = 63, post_days: int = 126):
    idx = returns_df.index
    if start_date not in idx:
        start_date = idx[idx.searchsorted(start_date)]
    loc = idx.get_loc(start_date)
    lo = max(0, loc - pre_days)
    hi = min(len(idx), loc + post_days + 1)
    window_idx = idx[lo:hi]

    local_nav = nav_df.loc[window_idx].copy()
    local_nav = local_nav / local_nav.iloc[0]

    local_rel = rel_payoff_df.loc[window_idx].copy()
    local_rel = local_rel - local_rel.iloc[0]

    plt.figure(figsize=(12, 5))
    local_nav.plot(ax=plt.gca())
    plt.axvline(start_date, linestyle="--", linewidth=1)
    plt.title(f"Event window NAV rebased around {pd.Timestamp(start_date).date()}")
    plt.ylabel("Rebased NAV")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(12, 5))
    local_rel.plot(ax=plt.gca())
    plt.axvline(start_date, linestyle="--", linewidth=1)
    plt.axhline(0.0, linestyle="--", linewidth=1)
    plt.title(f"Event window relative hedge payoff around {pd.Timestamp(start_date).date()}")
    plt.ylabel("Change in relative payoff vs window start")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


worst_windows = worst_forward_windows(returns_df["No hedge"], h=63, top_n=8, min_gap=63)
display(worst_windows)
worst_windows.to_csv(OUTPUT_DIR / "worst_forward_windows.csv", index=False)

# Plot the top 3 worst windows.
for dt in worst_windows["start"].head(3):
    plot_event_window(dt, pre_days=63, post_days=126)


## 12. Interpretation helper

以下は、今回の設計が狙い通りかを確認する簡易チェックです。

理想的には、

\[
DDC^F_{1D} > DDC^S_{1D}
\]

\[
DDC^S_{1Q} > DDC^F_{1Q}
\]

\[
\Gamma^F_{1D} > \Gamma^S_{1D}
\]

\[
\Gamma^S_{1Q} > \Gamma^F_{1Q}
\]

に近い関係が見えるはずです。

Combined については、

\[
DDC^C_{1D} \approx DDC^F_{1D}, \quad DDC^C_{1Q} \approx DDC^S_{1Q}
\]

および

\[
\Gamma^C_{1D} \approx \Gamma^F_{1D}, \quad \Gamma^C_{1Q} \approx \Gamma^S_{1Q}
\]

に近づけば、短期・中期の hedge payoff を合成できていると評価できます。ただし、sleeve weighted combination では convexity が平均化され、veto / min 型では convexity が強く出る一方で cash drag と whipsaw が増える可能性があります。


In [ ]:
def extract_metric(df: pd.DataFrame, strategy: str, horizon: str, col: str) -> float:
    sub = df[(df["Strategy"] == strategy) & (df["Horizon"] == horizon)]
    return sub[col].iloc[0] if len(sub) else np.nan

checks = []
checks.append({
    "Check": "First DDC 1D > Second DDC 1D",
    "Left": extract_metric(ddc_df, "First responder", "1D", "DDC"),
    "Right": extract_metric(ddc_df, "Second responder", "1D", "DDC"),
})
checks.append({
    "Check": "Second DDC 1Q > First DDC 1Q",
    "Left": extract_metric(ddc_df, "Second responder", "1Q", "DDC"),
    "Right": extract_metric(ddc_df, "First responder", "1Q", "DDC"),
})
checks.append({
    "Check": "First Convexity 1D > Second Convexity 1D",
    "Left": extract_metric(piecewise_convexity_df, "First responder", "1D", "Incremental Convexity"),
    "Right": extract_metric(piecewise_convexity_df, "Second responder", "1D", "Incremental Convexity"),
})
checks.append({
    "Check": "Second Convexity 1Q > First Convexity 1Q",
    "Left": extract_metric(piecewise_convexity_df, "Second responder", "1Q", "Incremental Convexity"),
    "Right": extract_metric(piecewise_convexity_df, "First responder", "1Q", "Incremental Convexity"),
})

check_df = pd.DataFrame(checks)
check_df["Pass"] = check_df["Left"] > check_df["Right"]
display(check_df)
check_df.to_csv(OUTPUT_DIR / "design_checks.csv", index=False)


## 13. Save consolidated outputs

このセルを実行すると、主要テーブルが `analysis_outputs/` に保存されます。チャートは必要に応じて `plt.savefig(...)` を追加してください。


In [ ]:
nav_df.to_csv(OUTPUT_DIR / "nav.csv")
dd_df.to_csv(OUTPUT_DIR / "drawdowns.csv")
rel_payoff_df.to_csv(OUTPUT_DIR / "relative_hedge_payoff.csv")
g_df.to_csv(OUTPUT_DIR / "exposure_multipliers.csv")
turnover_df.to_csv(OUTPUT_DIR / "turnover.csv")

with pd.ExcelWriter(OUTPUT_DIR / "signal_based_hedge_analysis_tables.xlsx") as writer:
    stats_df.to_excel(writer, sheet_name="performance")
    ddc_df.to_excel(writer, sheet_name="ddc", index=False)
    quad_convexity_df.to_excel(writer, sheet_name="quadratic_convexity", index=False)
    piecewise_convexity_df.to_excel(writer, sheet_name="piecewise_convexity", index=False)
    tail_payoff_ratio_df.to_excel(writer, sheet_name="tail_payoff_ratio", index=False)
    bucket_payoff_df.to_excel(writer, sheet_name="payoff_buckets", index=False)
    ddc_map.to_excel(writer, sheet_name="ddc_map", index=False)
    conv_map.to_excel(writer, sheet_name="convexity_map", index=False)
    worst_windows.to_excel(writer, sheet_name="worst_windows", index=False)
    check_df.to_excel(writer, sheet_name="design_checks", index=False)

print(f"Saved outputs to: {OUTPUT_DIR.resolve()}")
